# TaxaLens Foundation Corpus v0.6 — источники

Этот notebook подготавливает официальные источники, а не API-пилот: iNaturalist Open Data, selective BIOSCAN Diptera 30k, GBIF preserved specimens и DiSSCo. Всё тяжёлое сохраняется в Google Drive. Запускай секции по одной.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, subprocess, sys
REPO_URL = 'https://github.com/SaniyaSani/TaxaLens.git'
PROJECT = Path('/content/TaxaLens')
STORE = Path('/content/drive/MyDrive/TaxaLens/Foundation_v06')
if not PROJECT.exists(): subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT)], check=True)
else: subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
os.chdir(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-foundation.txt'], check=True)
for name in ('raw/inaturalist','raw/bioscan','raw/gbif','raw/dissco','manifests'): (STORE/name).mkdir(parents=True, exist_ok=True)
print('Persistent corpus store:', STORE)

## 1. iNaturalist Open Data
Скачивает официальный monthly metadata bundle и фильтрует все Research Grade Diptera. Это метаданные; фотографии пока не складываются в Drive.

In [ ]:
INAT = STORE/'raw/inaturalist'
subprocess.run([sys.executable, 'scripts/download_inat_metadata.py', '--out-dir', str(INAT)], check=True)
subprocess.run([sys.executable, 'scripts/ingest_inat.py', '--observations', str(INAT/'observations.csv.gz'), '--photos', str(INAT/'photos.csv.gz'), '--taxa', str(INAT/'taxa.csv.gz'), '--observers', str(INAT/'observers.csv.gz'), '--out', str(STORE/'manifests/inat_raw.parquet'), '--quality-grades', 'research', '--image-size', 'large'], check=True)

## 2. BIOSCAN-5M
v0.6 получает официальные metadata и выбирает ровно 30 000 Diptera. Из удалённых ZIP по HTTP Range скачиваются только выбранные JPEG; полные image packages не нужны. Повтор ячейки продолжает прерванную загрузку.

In [ ]:
BIOSCAN = STORE/'raw/bioscan'
subprocess.run([sys.executable, 'scripts/run_bioscan_30k.py', '--root', str(BIOSCAN)], check=True)
bioscan_manifest = BIOSCAN/'bioscan_diptera_30k_manifest.parquet'
assert bioscan_manifest.exists(), bioscan_manifest
__import__('shutil').copy2(bioscan_manifest, STORE/'manifests/bioscan_raw.parquet')
print('BIOSCAN 30k READY:', STORE/'manifests/bioscan_raw.parquet')

## 3. GBIF preserved Diptera + StillImage
Создай бесплатный GBIF account. В Colab Secrets (значок ключа слева) добавь `GBIF_USER`, `GBIF_PASSWORD`, `GBIF_EMAIL`. Первый запуск отправляет асинхронный запрос; вставь полученный key в `GBIF_DOWNLOAD_KEY` и перезапусти ячейку позже.

In [ ]:
from google.colab import userdata
GBIF_DOWNLOAD_KEY = ''
for key in ('GBIF_USER','GBIF_PASSWORD','GBIF_EMAIL'):
    try: os.environ[key] = userdata.get(key)
    except Exception: pass
GBIF = STORE/'raw/gbif'
if not GBIF_DOWNLOAD_KEY:
    subprocess.run([sys.executable, 'scripts/gbif_download.py', 'request', '--out', str(GBIF/'request.json'), '--submit'], check=True)
    print('Скопируй download key из строки выше в GBIF_DOWNLOAD_KEY.')
else:
    subprocess.run([sys.executable, 'scripts/gbif_download.py', 'status', GBIF_DOWNLOAD_KEY], check=True)
    try:
        subprocess.run([sys.executable, 'scripts/gbif_download.py', 'fetch', GBIF_DOWNLOAD_KEY, '--out', str(GBIF/'gbif_download.zip'), '--extract', str(GBIF/'extracted')], check=True)
        subprocess.run([sys.executable, 'scripts/ingest_gbif.py', '--occurrence', str(GBIF/'extracted/occurrence.txt'), '--multimedia', str(GBIF/'extracted/multimedia.txt'), '--out', str(STORE/'manifests/gbif_raw.parquet')], check=True)
    except subprocess.CalledProcessError:
        print('GBIF download ещё не готов; вернись к ячейке позже.')

## 4. DiSSCo openDS + Digital Media Objects
Публичный API выгружается страницами. Начни с 20 000 европейских specimen records; число можно увеличить. Скрипт разворачивает связанные media objects.

In [ ]:
DISSCO_MAX = 20000
DISSCO = STORE/'raw/dissco'
subprocess.run([sys.executable, 'scripts/download_dissco.py', '--out', str(DISSCO/'diptera_full.jsonl'), '--max-records', str(DISSCO_MAX), '--resume'], check=True)
subprocess.run([sys.executable, 'scripts/ingest_dissco.py', '--input', str(DISSCO/'diptera_full.jsonl'), '--out', str(STORE/'manifests/dissco_raw.parquet')], check=True)

## 5. Master manifest
Эта ячейка намеренно откажется продолжать, пока нет всех четырёх источников. Затем она объединит, удалит cross-source duplicates и назначит group-safe train/val/test split.

In [ ]:
sources = [STORE/'manifests/inat_raw.parquet', STORE/'manifests/bioscan_raw.parquet', STORE/'manifests/gbif_raw.parquet', STORE/'manifests/dissco_raw.parquet']
missing = [str(path) for path in sources if not path.exists()]
assert not missing, 'Не готовы источники: ' + ', '.join(missing)
combined = STORE/'manifests/combined.parquet'
dedup = STORE/'manifests/deduplicated.parquet'
master = STORE/'manifests/master_manifest.parquet'
subprocess.run([sys.executable, 'scripts/build_master_manifest.py', *map(str, sources), '--out', str(combined)], check=True)
subprocess.run([sys.executable, 'scripts/deduplicate_records.py', '--input', str(combined), '--out', str(dedup)], check=True)
subprocess.run([sys.executable, 'scripts/build_master_manifest.py', str(dedup), '--out', str(master)], check=True)
subprocess.run([sys.executable, 'scripts/corpus_report.py', '--input', str(master), '--out-json', str(STORE/'manifests/master_report.json'), '--out-md', str(STORE/'manifests/master_report.md')], check=True)
print('MASTER READY:', master)